# Task 3 (Dipin): CycleGAN photo â†” Monet: training analysis and evaluation

Domain **A = photos** (7,038), domain **B = Monet paintings** (300), from the Kaggle *I'm Something of a Painter Myself* data.
The model is trained by `train.py`, evaluated by `evaluate.py`, and the Kaggle archive is built by `make_submission.py`.
This notebook reads their outputs (the raw JSONL training log, `eval_metrics.json`, per-image CSVs, and human-audit sheets), then produces
every plot in `outputs/plots/` and the final `metrics_report.csv`. Run it from `task3_gan/dipin/src`.

In [ ]:
import json, csv, sys, glob
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Image as IPImage

sys.path.insert(0, '.')
from data import DIPIN, REPO

OUT = DIPIN / 'outputs'; PLOTS = OUT / 'plots'; PLOTS.mkdir(parents=True, exist_ok=True)
CFG = json.loads((DIPIN / 'configs' / 'task3_config.json').read_text())

# reference categorical palette (fixed order) and recessive ink
S1, S2, S3, S4 = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
    'axes.edgecolor': INK2, 'axes.labelcolor': INK2, 'xtick.color': INK2, 'ytick.color': INK2,
    'axes.titlecolor': INK, 'axes.titlesize': 11, 'axes.titleweight': 'bold', 'font.size': 9,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.spines.top': False,
    'axes.spines.right': False, 'lines.linewidth': 2, 'legend.frameon': False})

## 1. Raw training log

In [ ]:
log_path = sorted((OUT / 'logs').glob('task3_dipin_full_*.jsonl'))[-1]
recs = [json.loads(l) for l in open(log_path)]
start = next(r for r in recs if r['event'] == 'start')
steps = pd.DataFrame([r for r in recs if r['event'] == 'step'])
epochs = pd.DataFrame([r for r in recs if r['event'] == 'epoch']).drop_duplicates('epoch', keep='last')
end = next((r for r in recs if r['event'] == 'end'), None)
print('raw log:', log_path.relative_to(DIPIN), '|', len(steps), 'step records,', len(epochs), 'epochs')
print('hardware:', start['hardware'])
print('parameters:', start['params'])
epochs.tail(3).T

## 2. Loss curves (generator, discriminator, cycle, identity)

Thin lines are the 100-step window means from the log; bold lines are an exponential moving average (span 10 windows). Each panel has its own y-axis, so no quantity shares a scale with a different one.

In [ ]:
def smooth(s, w=10):
    return s.ewm(span=w).mean()

def curve(ax, x, y, color, label):
    ax.plot(x, y, color=color, alpha=0.25, linewidth=1)
    ax.plot(x, smooth(y), color=color, label=label)

kimg = steps['step'] * CFG['training']['batch_size'] / 1000
fig, axs = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
curve(axs[0, 0], kimg, steps['loss_G'], S1, 'total G loss')
axs[0, 0].set_title('Generator total loss  (adv + 10Â·cyc + 5Â·id)')
curve(axs[0, 1], kimg, steps['loss_G_adv'], S1, 'G adversarial (both directions)')
curve(axs[0, 1], kimg, steps['loss_D_A'], S2, 'D_A (photo critic)')
curve(axs[0, 1], kimg, steps['loss_D_B'], S3, 'D_B (Monet critic)')
axs[0, 1].axhline(0.25, color=INK2, linewidth=1, linestyle=':')
axs[0, 1].text(kimg.iloc[len(kimg) // 2], 0.26, 'D loss at LSGAN equilibrium (D = 0.5 everywhere): 0.25', color=INK2, va='bottom', ha='center', fontsize=8)
axs[0, 1].set_title('Adversarial losses (LSGAN / MSE)'); axs[0, 1].legend(loc='upper right')
curve(axs[1, 0], kimg, steps['loss_cyc'], S1, 'cycle L1 (Aâ†’Bâ†’A + Bâ†’Aâ†’B)')
axs[1, 0].set_title('Cycle-consistency loss (unweighted)')
curve(axs[1, 1], kimg, steps['loss_id'], S1, 'identity L1')
axs[1, 1].set_title('Identity loss (unweighted)')
for ax in axs[1]: ax.set_xlabel('training images seen (thousands)')
for ax in axs.flat: ax.set_ylim(bottom=0)
fig.suptitle('CycleGAN training losses', fontweight='bold', color=INK)
fig.savefig(PLOTS / 'loss_curves.png'); plt.show()

## 3. Stability: discriminator outputs, gradient norms, NaN count, learning rate

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, d, name in [(axs[0, 0], 'D_A', 'D_A (photo domain)'), (axs[0, 1], 'D_B', 'D_B (Monet domain)')]:
    curve(ax, kimg, steps[f'{d}_real'], S1, 'mean D(real)')
    curve(ax, kimg, steps[f'{d}_fake'], S2, 'mean D(fake from pool)')
    ax.axhline(0.5, color=INK2, linewidth=1, linestyle=':')
    ax.set_title(f'{name} patch outputs'); ax.legend(loc='best')
curve(axs[1, 0], kimg, steps['grad_norm_G'], S1, 'generators (G_AB + G_BA)')
curve(axs[1, 0], kimg, steps['grad_norm_D'], S2, 'discriminators (D_A + D_B)')
axs[1, 0].set_yscale('log'); axs[1, 0].set_title('Global gradient L2 norm (log scale)'); axs[1, 0].legend()
axs[1, 1].plot(kimg, steps['lr'] * 1e4, color=S1); axs[1, 1].set_ylabel('learning rate (×1e-4)'); axs[1, 1].set_ylim(bottom=0)
axs[1, 1].set_title(f"Learning rate (NaN/Inf steps skipped: {int(steps['nan_count_cum'].iloc[-1])})")
for ax in axs[1]: ax.set_xlabel('training images seen (thousands)')
fig.suptitle('Training stability', fontweight='bold', color=INK)
fig.savefig(PLOTS / 'stability.png'); plt.show()

stab = pd.DataFrame({
    'grad_norm_G': steps['grad_norm_G'].describe(), 'grad_norm_D': steps['grad_norm_D'].describe()}).T
stab['nan_inf_steps'] = int(epochs['nan_count_total'].iloc[-1])
stab

### Convergence per epoch on fixed held-out images
Every epoch, `train.py` translates the same 4 test photos and 4 test Monet paintings and records their cycle-reconstruction L1 (model scale [-1, 1]).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))
axs[0].plot(epochs['epoch'], epochs['fixed_test_cycle_L1_A'], color=S1, marker='o', markersize=3, label='photo â†’ Monet â†’ photo')
axs[0].plot(epochs['epoch'], epochs['fixed_test_cycle_L1_B'], color=S2, marker='o', markersize=3, label='Monet â†’ photo â†’ Monet')
axs[0].set_title('Held-out cycle L1 per epoch'); axs[0].set_xlabel('epoch'); axs[0].legend(); axs[0].set_ylim(bottom=0)
axs[1].plot(epochs['epoch'], epochs['loss_D_A'], color=S2, label='D_A'); axs[1].plot(epochs['epoch'], epochs['loss_D_B'], color=S3, label='D_B')
axs[1].plot(epochs['epoch'], epochs['loss_G_adv'] / 2, color=S1, label='G adversarial / 2 (per direction)')
axs[1].set_title('Epoch-mean adversarial losses'); axs[1].set_xlabel('epoch'); axs[1].legend(); axs[1].set_ylim(bottom=0)
fig.savefig(PLOTS / 'epoch_convergence.png'); plt.show()
epochs[['epoch', 'loss_G', 'loss_cyc', 'loss_id', 'loss_D_A', 'loss_D_B', 'fixed_test_cycle_L1_A',
        'fixed_test_cycle_L1_B', 'images_per_sec', 'epoch_seconds']].iloc[[0, 4, 9, 24, 34, 44, -1]].round(4)

## 4. Visual progression
Rows: test photo, photoâ†’Monet, reconstruction, test Monet, Monetâ†’photo, reconstruction.

In [ ]:
n_ep = int(epochs['epoch'].max())
for e in sorted({1, 5, 10, 25, n_ep}):
    p = OUT / 'samples' / f'epoch_{e:03d}.png'
    if p.exists():
        print(f'epoch {e}'); display(IPImage(filename=str(p), width=520))

## 5. Quantitative evaluation (both directions)
See `evaluate.py` for the protocol. A2B inputs are the 500 held-out photos, scored against all 300 real Monet. B2A inputs are all 300 Monet for the distribution metrics, scored against the 500 held-out real photos; the per-image metrics use only the 30 held-out Monet.
*Untranslated baseline* scores the raw inputs against the target domain; a working translator must land well below it.

In [ ]:
ev = json.loads((OUT / 'eval' / 'eval_metrics.json').read_text())
rows = ['FID', 'FID_untranslated_baseline', 'KID', 'KID_std', 'KID_untranslated_baseline', 'MiFID_estimate',
        'memorization_distance', 'precision', 'recall', 'density', 'coverage', 'cycle_L1', 'cycle_PSNR',
        'LPIPS_input_vs_translation', 'LPIPS_input_vs_reconstruction', 'content_cosine', 'translation_L1',
        'inference_images_per_sec', 'n_fake', 'n_real_ref', 'n_per_image']
table = pd.DataFrame({d: {k: ev[d][k] for k in rows} for d in ['A2B', 'B2A']})
table.columns = ['A2B photoâ†’Monet', 'B2A Monetâ†’photo']
table.round(4)

## 6. Cycle-consistency verification
We check the constraint three ways:
1. **Formula check.** Recompute the training cycle loss by hand on held-out images, using `F(G(x))` for both directions, and confirm it matches `nn.L1Loss`. This shows each generator is composed with its inverse, not with itself.
2. **Reference comparison.** Trained reconstructions should be far closer to the input than (a) the translation is, and (b) a reconstruction by randomly initialised generators is.
3. **Per-image check.** Reconstruction error should be below translation error for every image.

In [ ]:
import torch
from evaluate import load_generator, translate
from data import load_split, make_splits, to_model_range
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
G_AB = load_generator(DIPIN / 'checkpoints' / 'G_AB.pt', dev); G_BA = load_generator(DIPIN / 'checkpoints' / 'G_BA.pt', dev)
splits = make_splits(CFG)
A8, _ = load_split(splits, 'A', 'test'); B8, _ = load_split(splits, 'B', 'test')
a = to_model_range(A8[:8]).to(dev); b = to_model_range(B8[:8]).to(dev)
with torch.no_grad():
    rec_a, rec_b = G_BA(G_AB(a)), G_AB(G_BA(b))
    manual = (rec_a - a).abs().mean() + (rec_b - b).abs().mean()
    lib = torch.nn.L1Loss()(rec_a, a) + torch.nn.L1Loss()(rec_b, b)
    wrong_a = (G_AB(G_AB(a)) - a).abs().mean()  # composing a generator with itself breaks the cycle
print(f'manual cycle loss {manual.item():.5f} | nn.L1Loss {lib.item():.5f} | match: {torch.allclose(manual, lib)}')
print(f'A: L1(F(G(a)), a) = {(rec_a - a).abs().mean().item():.4f}   vs   L1(G(G(a)), a) = {wrong_a.item():.4f}')
cc = ev['cycle_check']
print(f"cycle L1 on first 100 test photos [0,1] scale: trained {cc['trained_cycle_L1_A_first100']:.4f}  vs random-init generators {cc['random_init_generators_cycle_L1_A']:.4f}")
pi = {d: pd.read_csv(OUT / 'eval' / f'per_image_{d}.csv') for d in ['A2B', 'B2A']}
for d, df in pi.items():
    print(f"{d}: reconstruction closer than translation for {(df.cycle_L1 < df.translation_L1).mean()*100:.1f}% of images; "
          f"mean cycle L1 {df.cycle_L1.mean():.4f} vs translation L1 {df.translation_L1.mean():.4f}; PSNR {df.cycle_PSNR.mean():.2f} dB")

## 7. Per-image metric distributions (held-out inputs)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3.3))
for ax, col, title in [(axs[0], 'cycle_L1', 'Cycle reconstruction L1 ([0,1] scale)'),
                       (axs[1], 'LPIPS_input_vs_translation', 'LPIPS input vs translation'),
                       (axs[2], 'content_cosine', 'Content cosine (Inception features)')]:
    ax.hist(pi['A2B'][col], bins=30, color=S1, alpha=0.85, label='A2B (500 photos)', density=True, edgecolor='white', linewidth=0.5)
    ax.hist(pi['B2A'][col], bins=12, color=S2, alpha=0.7, label='B2A (30 Monet)', density=True, edgecolor='white', linewidth=0.5)
    ax.set_title(title); ax.set_ylabel('density')
axs[0].legend()
fig.savefig(PLOTS / 'per_image_distributions.png'); plt.show()

## 8. Failure-case candidates
The held-out photos with the **lowest content cosine** (content lost), **highest cycle L1** (cycle constraint broken), and **highest LPIPS** (largest perceptual change) are selected automatically, then inspected by eye for `failure_analysis.md`.

In [ ]:
TR = OUT / 'translations'; A_dir = REPO / splits['A_dir']
def gallery(names, title, fname):
    fig, axs = plt.subplots(3, len(names), figsize=(2.2 * len(names), 6.9))
    for j, n in enumerate(names):
        stem = Path(n).stem
        for i, (img, lab) in enumerate([(A_dir / n, 'input'), (TR / 'A2B_photo_to_monet' / f'{stem}.jpg', 'photoâ†’Monet'),
                                         (TR / 'A2B_cycle_reconstruction' / f'{stem}.jpg', 'reconstruction')]):
            axs[i, j].imshow(Image.open(img)); axs[i, j].set_xticks([]); axs[i, j].set_yticks([]); axs[i, j].grid(False)
            if j == 0: axs[i, j].set_ylabel(lab)
        axs[0, j].set_title(stem, fontsize=8, fontweight='normal')
    fig.suptitle(title, fontweight='bold', color=INK); fig.savefig(PLOTS / fname); plt.show()

A = pi['A2B']
worst = {'lowest_content_cosine': A.nsmallest(5, 'content_cosine'), 'highest_cycle_L1': A.nlargest(5, 'cycle_L1'),
         'highest_LPIPS': A.nlargest(5, 'LPIPS_input_vs_translation')}
for k, df in worst.items():
    gallery(list(df.image), k.replace('_', ' '), f'failures_{k}.png'); display(df.round(4))
gallery(list(A.nlargest(5, 'content_cosine').image), 'best content preservation', 'best_content_cosine.png')
fails = pd.concat([df.assign(criterion=k) for k, df in worst.items()])
fails.to_csv(OUT / 'eval' / 'failure_candidates.csv', index=False)

## 9. Blinded human audit (30 fixed samples, 2 raters)
`human_audit.py prepare` writes the anonymised panels. Each rater fills in `ratings_rater1.csv` / `ratings_rater2.csv` independently, then `human_audit.py score` computes agreement.

In [ ]:
ap = OUT / 'human_audit' / 'audit_results.json'
audit = json.loads(ap.read_text()) if ap.exists() else None
if audit:
    display(pd.DataFrame(audit['criteria']).T.drop(columns='mean_by_direction').round(3))
    print('overall audit score (1-5):', round(audit['overall_audit_score'], 3), '| mean exact agreement %:',
          round(audit['overall_exact_agreement_pct'], 1), '| mean quadratic kappa:', round(audit['overall_quadratic_kappa'], 3))
else:
    print('audit not scored yet: rating sheets are pending')

## 10. Kaggle leaderboard

In [ ]:
kp = OUT / 'kaggle_submission' / 'leaderboard.json'
kaggle = json.loads(kp.read_text()) if kp.exists() else None
sub = json.loads((OUT / 'kaggle_submission' / 'submission_info.json').read_text())
print('submission:', sub['n_images'], 'images | zip sha256', sub['zip_sha256'][:16], '| G_AB sha256', sub['generator_sha256'][:16])
print(kaggle if kaggle else 'leaderboard score not recorded yet')

## 11. Resource usage and the final metrics report

In [ ]:
ts = json.loads((OUT / 'train_summary.json').read_text())
peak = float(epochs['peak_memory_gb'].max())
last = epochs.iloc[-1]
m = []
def add(d, k, v, note=''): m.append((d, k, v, note))
for d in ['A2B', 'B2A']:
    e = ev[d]
    add(d, 'FID', e['FID'], f"untranslated baseline {e['FID_untranslated_baseline']:.2f}; n_fake={e['n_fake']}, n_real={e['n_real_ref']}")
    add(d, 'KID', e['KID'], f"std {e['KID_std']:.4f} over 100 subsets; baseline {e['KID_untranslated_baseline']:.4f}")
    add(d, 'generative_precision', e['precision'], 'Kynkaanniemi 2019, k=3, Inception pool3')
    add(d, 'generative_recall', e['recall'], 'Kynkaanniemi 2019, k=3')
    add(d, 'density', e['density'], 'Naeem 2020, k=5')
    add(d, 'coverage', e['coverage'], 'Naeem 2020, k=5')
    add(d, 'cycle_reconstruction_L1', e['cycle_L1'], f"[0,1] pixel scale, held-out n={e['n_per_image']}; PSNR {e['cycle_PSNR']:.2f} dB")
    add(d, 'LPIPS', e['LPIPS_input_vs_translation'], f"AlexNet LPIPS input vs translation; input vs reconstruction {e['LPIPS_input_vs_reconstruction']:.4f}")
    add(d, 'content_preservation_cosine_similarity', e['content_cosine'], 'Inception pool3 features, input vs translation')
    add(d, 'MiFID_estimate', e['MiFID_estimate'], f"Kaggle-style; memorization distance {e['memorization_distance']:.4f}")
add('overall', 'generator_loss', last['loss_G'], 'final-epoch mean, total = adv + 10*cyc + 5*id')
add('overall', 'generator_adversarial_loss', last['loss_G_adv'], 'final-epoch mean, sum of both directions')
add('overall', 'discriminator_loss', (last['loss_D_A'] + last['loss_D_B']) / 2, f"final-epoch mean; D_A {last['loss_D_A']:.4f}, D_B {last['loss_D_B']:.4f}")
add('overall', 'cycle_consistency_loss', last['loss_cyc'], 'final-epoch mean, unweighted L1 sum of both cycles, [-1,1] scale')
add('overall', 'identity_loss', last['loss_id'], 'final-epoch mean, unweighted L1 sum of both directions')
add('overall', 'gradient_norm', last['grad_norm_G'], f"final-epoch mean G norm; D {last['grad_norm_D']:.3f}; max G over run {steps['grad_norm_G'].max():.2f}")
add('overall', 'nan_count', int(ts['nan_count_total']), 'non-finite loss/grad steps (skipped)')
add('overall', 'human_audit_score', audit['overall_audit_score'] if audit else '', '1-5 mean over style/content/artifacts, 2 raters' if audit else 'pending ratings')
add('overall', 'inter_rater_agreement', audit['overall_quadratic_kappa'] if audit else '',
    f"quadratic Cohen kappa; exact agreement {audit['overall_exact_agreement_pct']:.1f}%" if audit else 'pending ratings')
add('overall', 'parameter_count', ts['params']['total'], f"G_AB {ts['params']['G_AB']}, G_BA {ts['params']['G_BA']}, D_A {ts['params']['D_A']}, D_B {ts['params']['D_B']}")
add('overall', 'training_time', round(ts['train_hours'], 3), f"hours on {ts['hardware'].get('gpu', 'cpu')}; {ts['steps']} steps x batch {ts['batch_size']}")
add('overall', 'images_per_sec', round(ts['images_per_sec'], 2), f"training (paired A+B sample per image); inference A2B {ev['A2B']['inference_images_per_sec']:.1f} img/s")
add('overall', 'peak_memory_usage', round(peak, 3), 'GB, torch.cuda.max_memory_allocated during training')
add('overall', 'kaggle_public_score', kaggle.get('public_score', '') if kaggle else '', kaggle.get('note', 'MiFID, lower is better') if kaggle else 'pending submission')
add('overall', 'kaggle_private_score', kaggle.get('private_score', '') if kaggle else '', kaggle.get('rank_note', '') if kaggle else 'pending submission')
report = pd.DataFrame(m, columns=['direction', 'metric', 'value', 'notes'])
report.to_csv(DIPIN / 'metrics_report.csv', index=False)
report